# Module 1: From a quote tool to portfolio skills

Build an assistant with Gemini 3.5 Flash-Lite. Give it a Python tool for looking up a simulated stock price, then use an ADK skill to run a portfolio stress test. Finally, ask the assistant to draft a skill and try the draft after reviewing it.

A **tool** is a Python function the agent can call. A **skill** gives the agent instructions for a task and can make additional tools available when it loads.

Follow the [workshop setup](../README.md#getting-started), then open this notebook with the workshop's virtual environment. This lab was tested on Python 3.10 with Google ADK 2.8.0. Run the cells in order. Set `RUN_LIVE = True` in the next cell to call Gemini with the key in `.env.local`.

In [ ]:
import os
import sys
from math import isclose
from pathlib import Path

# Jupyter may start in the repository root or the notebook's directory.
here = Path.cwd().resolve()
project_root = next(
    (folder for folder in (here, *here.parents)
     if (folder / "module_01" / "portfolio_lab.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open the notebook from inside the workshop repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.skills import load_skill_from_dir
from google.adk.tools import FunctionTool

from module_01.financial_agent_app.agent import MODEL, build_agent
from module_01.financial_agent_app.portfolio import (
    calculate_shock, get_holdings, get_position_values, get_stock_price,
)
from module_01.financial_agent_app.skills import PORTFOLIO_STRESS_TEST, SKILL_AUTHOR
from module_01.portfolio_lab import ModelCallPacer, run_turn

RUN_LIVE = False
MODEL_CALL_INTERVAL = 5  # Seconds between model calls. Adjust for your quota.

if RUN_LIVE and not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise RuntimeError("Set GOOGLE_API_KEY in .env.local using the workshop setup instructions.")

pacer = ModelCallPacer(MODEL_CALL_INTERVAL)
print(f"Model: {MODEL} | Live calls enabled: {RUN_LIVE}")

## 1. Give an agent a quote tool

Call the quote function in Python first. Look up `LUMR` and then an unknown ticker. Next, build an ADK agent with the same function wrapped in `FunctionTool`. The agent can ask ADK to run that function and use its result in a reply.

In [ ]:
quote = get_stock_price("LUMR")
unknown_quote = get_stock_price("ZZZZ")

assert quote["price"] == 245.50 and quote["source"] == "simulated"
assert "error" in unknown_quote and "price" not in unknown_quote
print(quote)
print(unknown_quote)

quote_agent = Agent(
    name="quote_agent",
    model=MODEL,
    instruction=(
        "Use get_stock_price for quote requests. Label quotes as simulated "
        "and show dollar amounts to two decimal places. Report lookup errors."
    ),
    tools=[FunctionTool(get_stock_price)],
)

In [ ]:
async def ask(agent, prompt):
    # Each exercise starts a new session, with its own skill activation state.
    async with InMemoryRunner(
        agent=agent, app_name="module_1_notebook", plugins=[pacer],
    ) as runner:
        session = await runner.session_service.create_session(
            app_name=runner.app_name, user_id="learner",
        )
        return await run_turn(runner, session.user_id, session.id, prompt)

def tool_result(turn, name):
    return next(
        (event.response for event in turn.responses if event.name == name),
        None,
    )

def show_turn(turn):
    print("Tool calls:", " → ".join(call.name for call in turn.calls) or "none")
    for event in turn.responses:
        result = event.response or {}
        if event.name == "load_skill":
            print("Skill:", result.get("skill_name") or result.get("error"))
        elif event.name not in {"list_skills", "stage_skill_draft"}:
            print(f"{event.name}: {result}")
    print(f"\nAgent > {turn.text}")

In [ ]:
if RUN_LIVE:
    quote_turn = await ask(quote_agent, "What is the simulated price of LUMR?")
    show_turn(quote_turn)
else:
    print("Set RUN_LIVE = True in the setup cell to ask the quote agent.")

## 2. Load a skill for a portfolio stress test

The portfolio starts at USD 100,000. Its holdings are 20 LUMR shares at USD 245.50, 100 AAPL shares at USD 180, and USD 77,090 in cash. If LUMR falls 10% and the other values stay flat, the portfolio ends at USD 99,509.

Inspect the `portfolio-stress-test` skill. Its `adk_additional_tools` entry makes `calculate_shock` available once ADK loads the skill. Run the calculator directly to get a value you can compare with the live trace.

In [ ]:
print(PORTFOLIO_STRESS_TEST.frontmatter.model_dump(exclude_none=True))
print(PORTFOLIO_STRESS_TEST.instructions)

snapshot = get_holdings()
quotes = {symbol: get_stock_price(symbol)["price"] for symbol in snapshot["holdings"]}
expected_stress = calculate_shock(
    snapshot["holdings"], quotes, snapshot["cash"], {"LUMR": -10},
)
assert expected_stress == {
    "starting_value": 100_000,
    "ending_value": 99_509,
    "change_dollars": -491,
    "change_pct": -0.491,
}
print("\nLocal calculator:", expected_stress)

The [portfolio agent](financial_agent_app/agent.py) registers the simulated data tools and attaches a `SkillToolset`. In the live trace, find `load_skill` before `calculate_shock`. Compare the calculator's result with your local run.

Try changing the prompt to “What happens if LUMR drops?” as an exercise. A useful reply asks how far it drops before running the calculator.

In [ ]:
portfolio_agent = build_agent()
if RUN_LIVE:
    stress_turn = await ask(portfolio_agent, "What happens to my portfolio if LUMR drops 10%?")
    show_turn(stress_turn)
    actual_stress = tool_result(stress_turn, "calculate_shock")
    print("Matches the local calculator:", actual_stress == expected_stress)
else:
    print("Set RUN_LIVE = True to see the stress-test skill in the trace.")

## 3. Ask the assistant to draft a new skill

Ask the same portfolio assistant to create a target-weight skill. Its `skill-author` skill supplies an example and a template. The `stage_skill_draft` tool returns the proposed `SKILL.md` and Python function as text for you to review.

In [ ]:
print("Authoring skill:", SKILL_AUTHOR.frontmatter.description)
draft = None
if RUN_LIVE:
    author_turn = await ask(
        portfolio_agent,
        "Create a new ADK skill called target-weight for finding the price move "
        "needed to reach a target portfolio weight. Name its tool calculate_target_weight.",
    )
    show_turn(author_turn)
    draft = tool_result(author_turn, "stage_skill_draft")
    if draft and draft.get("status") == "drafted":
        print(f"\nDraft SKILL.md\n\n{draft['skill_markdown']}")
        print(f"\nDraft Python\n\n{draft['python_source']}")
    else:
        print("No draft was staged. Check the trace and try the request again.")
else:
    print("Set RUN_LIVE = True to ask the assistant to draft the skill.")

## 4. Review and try the new skill

Read both drafts. The skill should call `get_position_values` and pass its two values directly to the calculator. Python values the holdings and includes cash in the rest of the portfolio. The calculator uses those values and a target expressed as a fraction, such as `0.05`. Check that the function handles a zero position value and rejects invalid target fractions before dividing.

Save the reviewed Markdown as `SKILL.md` in a local directory named `target-weight`. In the next cell, set `REVIEWED_SKILL_DIR` to that directory, paste the reviewed Python function, and assign the function to `REVIEWED_TOOL`. The first check uses a USD 4,000 position and USD 95,000 elsewhere. The position needs a **25% rise** to reach a 5% weight.

In [ ]:
REVIEWED_SKILL_DIR = None  # Example: Path("/path/to/target-weight")
REVIEWED_TOOL = None

# Paste the reviewed definition of calculate_target_weight here.
# Then assign REVIEWED_TOOL = calculate_target_weight.

In [ ]:
reviewed_agent = None
if REVIEWED_SKILL_DIR is None and REVIEWED_TOOL is None:
    print("Fill in both values above after reviewing the draft.")
elif REVIEWED_SKILL_DIR is None or not callable(REVIEWED_TOOL):
    raise ValueError("Provide the reviewed skill directory and the Python function together.")
else:
    reviewed_skill = load_skill_from_dir(REVIEWED_SKILL_DIR)
    tool_names = reviewed_skill.frontmatter.metadata.get("adk_additional_tools", [])
    if tool_names != [REVIEWED_TOOL.__name__]:
        raise ValueError("The skill metadata must name the reviewed Python function.")

    check = REVIEWED_TOOL(4_000, 95_000, 0.05)
    assert isclose(check["required_move_pct"], 25.0), check
    reviewed_agent = build_agent(extra_skills=[reviewed_skill], extra_tools=[REVIEWED_TOOL])
    print(f"Local check passed: {check['required_move_pct']:.2f}%. Skill registered.")

In a fresh session, ask what price move would take LUMR to 5% of the simulated portfolio. The position is worth USD 4,910 and everything else is worth USD 95,090. Expect about **+1.929%**. Look for `target-weight` in the skill load response and `get_position_values` before the calculator call.

In [ ]:
print("Local LUMR valuation:", get_position_values("LUMR"))
if not RUN_LIVE:
    print("Set RUN_LIVE = True to try the reviewed skill with Gemini.")
elif reviewed_agent is None:
    print("Register the reviewed skill and Python function first.")
else:
    reuse_turn = await ask(
        reviewed_agent,
        "How much would LUMR have to rise to reach 5% of my portfolio? Keep everything else flat.",
    )
    show_turn(reuse_turn)

The [Module 1 README](README.md) shows how to run the same assistant from a terminal. In [Module 2](../module_02/README.md), connect a specialist to documents so it can answer questions about the firm's simulated investment policy.